# OrbitAvanya Email Campaign — with Open Tracking

**New Feature:** Each outgoing email now contains an invisible 1×1 tracking pixel.  
When a recipient **opens** the email, their email client loads that pixel from your tracking server,  
and `Email_Tracking.xlsx` is automatically updated with:
- `Email Open Status` → **Opened** (highlighted green)
- `Email Opened At`  → timestamp of first open
- `Open Count`       → how many times the email was opened

---
### Prerequisites
1. Start `email_tracker_server.py` on your machine.
2. Expose it via [ngrok](https://ngrok.com): `ngrok http 5050`
3. Copy the `https://xxxx.ngrok.io` URL into `Config.TRACKING_BASE_URL` below.

# ============================================================================
# SECTION 1: IMPORTS
# ============================================================================
import pandas as pd
import smtplib
from email.utils import formatdate, make_msgid
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import logging
import time
import os
import json
from datetime import datetime, timedelta
import random
from pathlib import Path
from tqdm import tqdm
import imaplib
import email
from email.header import decode_header
from urllib.parse import quote   # NEW — for encoding email in tracking URL

In [ ]:
# ============================================================================
# SECTION 2: LOGGING
# ============================================================================
log_directory = "email_campaign_logs"
Path(log_directory).mkdir(exist_ok=True)
log_filename = f"{log_directory}/email_campaign_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_filename),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)
logger.info("EMAIL CAMPAIGN AUTOMATION STARTED")

In [ ]:
# ============================================================================
# SECTION 3: CONFIGURATION
# ============================================================================
class Config:
    # ── Email credentials ────────────────────────────────────────────────────
    SENDER_EMAIL    = "pradeep@orbitavanyatech.com"
    SENDER_PASSWORD = "Prad@2026"               # Use App Password if Gmail
    SMTP_SERVER     = "mail.orbitavanyatech.com"
    SMTP_PORT       = 465

    # ── IMAP (reply tracking) ────────────────────────────────────────────────
    IMAP_SERVER = "mail.orbitavanyatech.com"
    IMAP_PORT   = 993

    # ── File paths ───────────────────────────────────────────────────────────
    EXCEL_FILE_PATH = r"C:\Users\Shree\Downloads\EmailAutomation\Trail for email automation (1).xlsx"
    SHEET_NAME      = "Sheet1"
    TRACKER_FILE    = r"C:\Users\Shree\Downloads\EmailAutomation\Email_Tracking.xlsx"
    DUPLICATES_FILE = "sent_emails_log.json"

    # ── Tracking pixel server ─────────────────────────────────────────────────
    # Step 1: run email_tracker_server.py
    # Step 2: run  ngrok http 5050
    # Step 3: paste your ngrok URL here (no trailing slash)
    TRACKING_BASE_URL = "https://abc123.ngrok.io"   # <── CHANGE THIS

    # ── Email subject ────────────────────────────────────────────────────────
    EMAIL_SUBJECT   = "Exploring Collaboration Opportunity with {company_name}"
    EMAIL_BODY      = """Dear {ceo_name},\n\nGreetings from OrbitAvanya Tech LLP.\n"""

    # ── Rate limiting ────────────────────────────────────────────────────────
    EMAIL_DELAY_MIN  = 20
    EMAIL_DELAY_MAX  = 40
    BATCH_SIZE       = 50
    SKIP_FAILED_EMAILS = False

In [ ]:
# ============================================================================
# SECTION 4: TRACKING PIXEL HELPER
# ============================================================================
def build_tracking_pixel(recipient_email: str, company_name: str) -> str:
    """
    Returns an <img> tag that, when loaded by the email client,
    hits your tracking server and marks the email as opened.

    The image is 1×1 pixels and fully transparent — completely invisible
    to the reader but triggers the open event.
    """
    encoded_email   = quote(recipient_email, safe="")
    encoded_company = quote(company_name,    safe="")
    tracking_url = (
        f"{Config.TRACKING_BASE_URL}/track"
        f"?email={encoded_email}&company={encoded_company}"
    )
    return (
        f'<img src="{tracking_url}" '
        f'width="1" height="1" border="0" '
        f'style="display:block;width:1px;height:1px;" alt="" />'
    )


def inject_tracking_pixel(html_body: str, recipient_email: str, company_name: str) -> str:
    """
    Inserts the tracking pixel just before the closing </body> tag.
    If no </body> found, appends it at the end.
    """
    pixel = build_tracking_pixel(recipient_email, company_name)
    if "</body>" in html_body.lower():
        idx = html_body.lower().rfind("</body>")
        return html_body[:idx] + pixel + html_body[idx:]
    return html_body + pixel


print("✅ Tracking pixel helper loaded.")
print(f"   Tracking URL base: {Config.TRACKING_BASE_URL}")

In [ ]:
# ============================================================================
# SECTION 5: DUPLICATE TRACKER
# ============================================================================
class DuplicateTracker:
    def __init__(self, file_path):
        self.file_path   = file_path
        self.sent_emails = self.load_sent_emails()

    def load_sent_emails(self):
        try:
            if os.path.exists(self.file_path):
                with open(self.file_path, 'r') as f:
                    data = json.load(f)
                    logger.info(f"Loaded {len(data)} previously sent email records")
                    return set(data)
            return set()
        except Exception as e:
            logger.error(f"Error loading duplicate file: {e}")
            return set()

    def save_sent_email(self, email_address):
        self.sent_emails.add(email_address)
        try:
            with open(self.file_path, 'w') as f:
                json.dump(list(self.sent_emails), f, indent=2)
        except Exception as e:
            logger.error(f"Error saving sent email record: {e}")

    def is_duplicate(self, email_address):
        return email_address in self.sent_emails

In [ ]:
# ============================================================================
# SECTION 6: EMAIL SENDER  (UPDATED — injects tracking pixel)
# ============================================================================
class EmailSender:
    def __init__(self, sender_email, sender_password, smtp_server, smtp_port):
        self.sender_email    = sender_email
        self.sender_password = sender_password
        self.smtp_server     = smtp_server
        self.smtp_port       = smtp_port

    def create_personalized_email(self, recipient_email, recipient_name,
                                   company_name, subject_template, body_template):
        subject = subject_template.format(
            ceo_name=recipient_name,
            company_name=company_name
        )
        body = body_template.format(
            ceo_name=recipient_name,
            company_name=company_name
        )
        return subject, {
            "ceo_name":    recipient_name,
            "company_name": company_name,
            "body_text":    body
        }

    def send_email(self, recipient_email, subject, body):
        try:
            message = MIMEMultipart("alternative")
            message["From"]            = f"Pradeep | Orbitavanya Tech LLP <{self.sender_email}>"
            message["To"]              = recipient_email
            message["Subject"]         = subject
            message["Reply-To"]        = "info@orbitavanyatech.com"
            message["Date"]            = formatdate(localtime=True)
            message["Message-ID"]      = make_msgid()
            message["List-Unsubscribe"]= "<mailto:pradeep@orbitavanyatech.com?subject=unsubscribe>"

            # Choose template based on subject
            if "reminder" in subject.lower():
                template_file = "Followup_template.html"
            else:
                template_file = "Email_Template_final_1.html"

            with open(template_file, "r", encoding="utf-8") as f:
                html_body = f.read()

            # Personalise placeholders
            html_body = html_body.replace("{{CEO_Name}}",     body["ceo_name"])
            html_body = html_body.replace("{{Company_Name}}", body["company_name"])

            # ── NEW: Inject tracking pixel ────────────────────────────────
            html_body = inject_tracking_pixel(
                html_body,
                recipient_email,
                body["company_name"]
            )
            # ─────────────────────────────────────────────────────────────

            plain_text = (
                f"Dear {body['ceo_name']},\n\n"
                "Greetings from OrbitAvanya Tech LLP.\n\n"
                "We would like to explore potential collaboration opportunities.\n\n"
                "Please visit: https://www.orbitavanyatech.com\n\n"
                "Regards,\nOrbitAvanya Tech LLP"
            )
            message.attach(MIMEText(plain_text, "plain"))
            message.attach(MIMEText(html_body,  "html"))

            with smtplib.SMTP_SSL(self.smtp_server, self.smtp_port) as server:
                server.login(self.sender_email, self.sender_password)
                server.sendmail(self.sender_email, recipient_email, message.as_string())

            logger.info(f"Email sent successfully to {recipient_email}")
            return True, "Email sent successfully"

        except Exception as e:
            logger.error(f"Failed to send email to {recipient_email}: {e}")
            return False, str(e)

In [ ]:
# ============================================================================
# SECTION 7: TRACKER FILE HELPER
# ============================================================================
def save_email_record(company, contact_person, email_addr, subject):
    tracker   = pd.read_excel(Config.TRACKER_FILE)
    sent_date = datetime.now()

    new_row = {
        "Company":          company,
        "Contact Person":   contact_person,
        "Email":            email_addr,
        "Subject":          subject,
        "Sent Date":        sent_date.strftime("%d-%m-%Y"),
        "Reply Received":   "No",
        "Reply Date":       "",
        "Reminder Due":     (sent_date + timedelta(days=10)).strftime("%d-%m-%Y"),
        "Reminder Sent":    "No",
        # ── NEW columns (blank at send time; server fills them on open)
        "Email Open Status": "Not Opened",
        "Email Opened At":   "",
        "Open Count":        0,
    }

    tracker = pd.concat([tracker, pd.DataFrame([new_row])], ignore_index=True)
    tracker.to_excel(Config.TRACKER_FILE, index=False)
    print(f" Email record saved for {email_addr}")

In [ ]:
# ============================================================================
# SECTION 8: CAMPAIGN RUNNER
# ============================================================================
class EmailCampaign:
    def __init__(self, config):
        self.config           = config
        self.email_sender     = EmailSender(
            config.SENDER_EMAIL, config.SENDER_PASSWORD,
            config.SMTP_SERVER,  config.SMTP_PORT
        )
        self.duplicate_tracker = DuplicateTracker(config.DUPLICATES_FILE)
        self.df      = None
        self.results = []

    def read_excel_file(self):
        try:
            logger.info(f"Reading Excel file: {self.config.EXCEL_FILE_PATH}")
            self.df = pd.read_excel(self.config.EXCEL_FILE_PATH, sheet_name=self.config.SHEET_NAME)
            logger.info(f"Loaded {len(self.df)} records")
            print("\nFirst few records:")
            print(self.df.head())
            return True
        except FileNotFoundError:
            logger.error(f"Excel file not found: {self.config.EXCEL_FILE_PATH}")
            return False
        except Exception as e:
            logger.error(f"Error reading Excel: {e}")
            return False

    def validate_data(self):
        required = ['CEO Name', 'Company Name', 'Lead Email ID']
        missing  = [c for c in required if c not in self.df.columns]
        if missing:
            logger.error(f"Missing columns: {missing}")
            return False
        return True

    def send_campaign(self):
        if not self.read_excel_file():  return False
        if not self.validate_data():    return False

        for col in ['Send Status', 'Send Message', 'Send Timestamp']:
            if col not in self.df.columns:
                self.df[col] = ''

        for index, row in tqdm(self.df.iterrows(), total=len(self.df), desc="Sending emails"):
            try:
                ceo_name        = str(row['CEO Name']).strip()
                company_name    = str(row['Company Name']).strip()
                recipient_email = str(row['Lead Email ID']).strip()

                if not recipient_email or recipient_email.lower() == 'nan':
                    self.df.at[index, 'Send Status']  = 'Skipped'
                    self.df.at[index, 'Send Message'] = 'No email address'
                    continue

                if self.duplicate_tracker.is_duplicate(recipient_email):
                    self.df.at[index, 'Send Status']  = 'Duplicate'
                    self.df.at[index, 'Send Message'] = 'Already sent'
                    continue

                subject, body = self.email_sender.create_personalized_email(
                    recipient_email, ceo_name, company_name,
                    self.config.EMAIL_SUBJECT, self.config.EMAIL_BODY
                )

                success, message = self.email_sender.send_email(recipient_email, subject, body)

                if success:
                    self.df.at[index, 'Send Status']     = 'Success'
                    self.df.at[index, 'Send Message']    = message
                    self.df.at[index, 'Send Timestamp']  = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                    self.duplicate_tracker.save_sent_email(recipient_email)
                    save_email_record(company_name, ceo_name, recipient_email, subject)
                else:
                    self.df.at[index, 'Send Status']  = 'Failed'
                    self.df.at[index, 'Send Message'] = message
                    if self.config.SKIP_FAILED_EMAILS:
                        break

                delay = random.uniform(self.config.EMAIL_DELAY_MIN, self.config.EMAIL_DELAY_MAX)
                time.sleep(delay)

            except Exception as e:
                logger.error(f"Error on row {index}: {e}")
                self.df.at[index, 'Send Status']  = 'Error'
                self.df.at[index, 'Send Message'] = str(e)

        return True

    def save_results_to_excel(self):
        try:
            output_filename = f"companies_data_RESULTS_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
            self.df.to_excel(output_filename, index=False, sheet_name=self.config.SHEET_NAME)
            print(f"\nResults saved: {output_filename}")
            print(self.df['Send Status'].value_counts())
            return True
        except Exception as e:
            logger.error(f"Error saving results: {e}")
            return False

In [ ]:
# ============================================================================
# SECTION 9: REPLY TRACKER (unchanged)
# ============================================================================
class ReplyTracker:
    def __init__(self):
        self.email       = Config.SENDER_EMAIL
        self.password    = Config.SENDER_PASSWORD
        self.imap_server = Config.IMAP_SERVER
        self.imap_port   = Config.IMAP_PORT
        self.mail        = None

    def connect(self):
        self.mail = imaplib.IMAP4_SSL(self.imap_server, self.imap_port)
        self.mail.login(self.email, self.password)
        print("Connected to IMAP Server")

    def load_tracker(self):
        self.tracker = pd.read_excel(Config.TRACKER_FILE)
        self.tracker["Email"] = self.tracker["Email"].astype(str).str.strip().str.lower()
        print("Tracker Loaded:", len(self.tracker))

    def check_replies(self):
        self.mail.select("INBOX")
        status, messages = self.mail.search(None, "ALL")
        mail_ids = messages[0].split()
        print("Total Emails in Inbox:", len(mail_ids))

        for mail_id in reversed(mail_ids):
            status, msg_data = self.mail.fetch(mail_id, "(RFC822)")
            for response in msg_data:
                if isinstance(response, tuple):
                    from email.utils import parseaddr
                    msg    = email.message_from_bytes(response[1])
                    sender = parseaddr(msg["From"])[1].strip().lower()
                    if sender in self.tracker["Email"].values:
                        print("Reply found from:", sender)
                        self.tracker.loc[self.tracker["Email"] == sender, "Reply Received"] = "Yes"
                        self.tracker.loc[self.tracker["Email"] == sender, "Reply Date"]     = datetime.now().strftime("%d-%m-%Y")

        self.tracker.to_excel(Config.TRACKER_FILE, index=False)

    def logout(self):
        if self.mail:
            self.mail.logout()
            print("Disconnected from IMAP Server")


def run_reply_tracker():
    rt = ReplyTracker()
    rt.connect()
    rt.load_tracker()
    rt.check_replies()
    rt.logout()

In [ ]:
# ============================================================================
# SECTION 10: REMINDER SENDER (unchanged)
# ============================================================================
def send_reminder_email(company, contact_person, email_addr, subject):
    reminder_subject = f"Gentle Reminder - {subject}"
    reminder_body    = {"ceo_name": contact_person, "company_name": company}
    sender = EmailSender(
        Config.SENDER_EMAIL, Config.SENDER_PASSWORD,
        Config.SMTP_SERVER,  Config.SMTP_PORT
    )
    return sender.send_email(email_addr, reminder_subject, reminder_body)


def run_reminder_sender():
    tracker = pd.read_excel(Config.TRACKER_FILE)
    today   = datetime.today().strftime("%d-%m-%Y")
    due     = tracker[(tracker["Reply Received"] == "No") & (tracker["Reminder Sent"] == "No")]
    print("Today:", today, "| Reminder candidates:", len(due))

    sender = EmailSender(
        Config.SENDER_EMAIL, Config.SENDER_PASSWORD,
        Config.SMTP_SERVER,  Config.SMTP_PORT
    )
    for index, row in due.iterrows():
        success, message = send_reminder_email(
            row["Company"], row["Contact Person"], row["Email"], row["Subject"]
        )
        if success:
            tracker.at[index, "Reminder Sent"] = "Yes"
            print(f"Reminder sent to {row['Email']}")
        else:
            print(f"Failed for {row['Email']}: {message}")

    tracker.to_excel(Config.TRACKER_FILE, index=False)
    print("Reminder Sender Completed!")

In [ ]:
# ============================================================================
# SECTION 11: RUN CAMPAIGN
# ============================================================================
def main():
    print("╔" + "=" * 68 + "╗")
    print("║" + " EMAIL CAMPAIGN WITH OPEN TRACKING ".center(68) + "║")
    print("╚" + "=" * 68 + "╝\n")

    # Warn if tracking URL is still the placeholder
    if "YOUR-NGROK-URL" in Config.TRACKING_BASE_URL:
        print("⚠️  WARNING: TRACKING_BASE_URL is not set in Config!")
        print("   Open tracking will NOT work until you set a real URL.\n")

    campaign = EmailCampaign(Config)
    if campaign.send_campaign():
        campaign.save_results_to_excel()
        print("\n✅ Campaign completed successfully!")
    else:
        print("\n❌ Campaign failed. Check logs.")

main()

In [ ]:
# ============================================================================
# OPTIONAL: Run reply tracker + reminder sender after campaign
# ============================================================================
# Uncomment when you want to check replies and send reminders:

# run_reply_tracker()
# run_reminder_sender()